# Acción Comercial: Estrategia de Retención (Enfoque Profesor)

**Objetivo:** Diseñar una estrategia de retención para **10,000 nuevos clientes** usando:
- Fórmula del profesor: **CF = BASE × (1 + α)^n**
- Archivo **Costes.csv** para BASE y Margen
- Modelo **XGBoost Enfoque 2** para probabilidades de churn
- Segmentación en **5 grupos** de riesgo
- **Response rate 10%** fijo

---

## Fórmulas Clave (según profesor)

1. **Coste Fijo (CF)**: `CF = Mantenimiento_medio × (1 + α)^(n+1)`
   - α = 7% para modelos A, B
   - α = 10% para modelos C-K
   - n = número de revisiones previas

2. **Precio que paga el cliente**: `Precio = CF / (1 - Margen/100)`

3. **Margen bruto**: `Margen_Bruto = Precio - CF`

4. **Coste de marketing**: `Coste_Marketing = CF × 0.01` (1% del CF)

5. **ROI**: `ROI = (Margen_Bruto - Costes_Campaña) / Costes_Campaña`
   - Costes_Campaña = Coste_Contacto + Coste_Marketing + Descuento
   - **NO incluir CF** en el denominador

## 1. Carga y Preparación de Datos

In [259]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

pd.options.display.float_format = '{:,.2f}'.format

In [260]:
# ── Cargar datos ──────────────────────────────────────────────
dm = pd.read_csv('Data/Datamart/datamart_final_v2.csv')
df_raw = pd.read_csv('Data/DataLake/customer_data.csv')
df_fresh = pd.read_csv('Data/DataLake/nuevos_clientes.csv')

# ── Cargar Costes.csv (BASE y Margen) ────────────────────────
costes = pd.read_csv('Data/DataLake/Costes.csv')

print(f"Datos de entrenamiento: {len(dm):,} clientes")
print(f"Nuevos clientes: {len(df_fresh):,} clientes")
print(f"\nCostes.csv (BASE y Margen por modelo):")
print(costes[['Modelo', 'Mantenimiento_medio', 'Margen']].to_string(index=False))

Datos de entrenamiento: 58,049 clientes
Nuevos clientes: 10,000 clientes

Costes.csv (BASE y Margen por modelo):
Modelo  Mantenimiento_medio  Margen
     A               250.00   28.00
     B               263.00   33.00
     C               276.00   33.00
     D               290.00   33.00
     E               305.00   37.00
     F               320.00   42.00
     G               336.00   42.00
     H               353.00   42.00
     I               371.00   43.00
     J               390.00    5.00
     K               410.00    5.00


In [261]:
# ── Construir Churn_Final (Enfoque 2) ────────────────────────
df_raw['Sales_Date'] = pd.to_datetime(df_raw['Sales_Date'], format='%d/%m/%Y', errors='coerce')
fecha_corte = pd.to_datetime('31/12/2023', format='%d/%m/%Y')
df_raw['antiguedad_dias'] = (fecha_corte - df_raw['Sales_Date']).dt.days.clip(lower=0)
df_raw['Churn_bin'] = (df_raw['Churn_400'] == 'Y').astype(int)

dm['Churn_Final'] = (
    (df_raw['Churn_bin'].values == 1) |
    ((df_raw['Revisiones'].values == 0) & (df_raw['antiguedad_dias'].values > 400))
).astype(int)

print(f"Tasa de churn: {dm['Churn_Final'].mean()*100:.1f}%")

Tasa de churn: 33.3%


## 2. Entrenamiento XGBoost y Predicción

In [262]:
# ── Features del Enfoque 2 ────────────────────────────────────
FEATURES_NUMERICAS = [
    'km_ultima_revision', 'PVP', 'Edad', 'RENTA_MEDIA_ESTIMADA',
    'gasto_relativo', 'Kw', 'Margen_eur_bruto', 'Margen_eur',
    'ENCUESTA_CLIENTE_ZONA_TALLER'
]
FEATURES_BINARIAS = [
    'tiene_queja', 'en_garantia_bin', 'MANTENIMIENTO_GRATUITO',
    'Lead_compra', 'seguro_bateria_bin', 'sin_encuesta', 'origen_internet'
]
FEATURES_CATEGORICAS = [
    'Modelo', 'ZONA', 'FORMA_PAGO', 'TIPO_CARROCERIA',
    'Equipamiento', 'Fuel', 'TRANSMISION_ID'
]
ALL_FEATURES = FEATURES_NUMERICAS + FEATURES_BINARIAS + FEATURES_CATEGORICAS
COLS_DROP = ['Churn_bin', 'Churn_Final', 'Churn_Corregido', 'perfil_cliente']

X = dm.drop(columns=[c for c in COLS_DROP if c in dm.columns])
y = dm['Churn_Final']

# Label Encoders
label_encoders = {}
for col in FEATURES_CATEGORICAS:
    le = LabelEncoder()
    le.fit(df_raw[col].astype(str))
    label_encoders[col] = le

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
ratio = (y_train == 0).sum() / (y_train == 1).sum()

# Entrenar XGBoost
xgb_model = XGBClassifier(
    n_estimators=200, max_depth=3, min_child_weight=10,
    gamma=1.0, reg_alpha=1.0, reg_lambda=5.0,
    learning_rate=0.05, scale_pos_weight=ratio,
    random_state=42, eval_metric='logloss', n_jobs=-1
)
xgb_model.fit(X_train, y_train)
print("Modelo XGBoost entrenado")

Modelo XGBoost entrenado


In [263]:
# ── Preprocesar nuevos clientes ──────────────────────────────
fresh = df_fresh.copy()

fresh['QUEJA'] = fresh['QUEJA'].fillna('NO')
fresh['tiene_queja'] = (fresh['QUEJA'].str.upper().str.strip() == 'SI').astype(int)
fresh['en_garantia_bin'] = (fresh['EN_GARANTIA'].fillna('NO').str.upper().str.strip() == 'SI').astype(int)
fresh['gasto_relativo'] = fresh['PVP'] / fresh['RENTA_MEDIA_ESTIMADA'].clip(lower=1)
fresh['seguro_bateria_bin'] = (fresh['SEGURO_BATERIA_LARGO_PLAZO'].fillna('NO').str.upper().str.strip() == 'SI').astype(int)
fresh['sin_encuesta'] = fresh['ENCUESTA_CLIENTE_ZONA_TALLER'].isnull().astype(int)
fresh['ENCUESTA_CLIENTE_ZONA_TALLER'] = fresh['ENCUESTA_CLIENTE_ZONA_TALLER'].fillna(0)
fresh['origen_internet'] = (fresh['Origen'].fillna('Tienda').str.strip() == 'Internet').astype(int)

fresh_feat = fresh[ALL_FEATURES].copy()

for col in FEATURES_NUMERICAS:
    if col == 'ENCUESTA_CLIENTE_ZONA_TALLER':
        continue
    if fresh_feat[col].isnull().sum() > 0:
        fresh_feat[col] = fresh_feat[col].fillna(fresh_feat[col].median())

for col in FEATURES_CATEGORICAS:
    if fresh_feat[col].isnull().sum() > 0:
        fresh_feat[col] = fresh_feat[col].fillna(fresh_feat[col].mode()[0])
    le = label_encoders[col]
    known = set(le.classes_)
    fresh_feat[col] = fresh_feat[col].astype(str).apply(
        lambda x, k=known, enc=le: enc.transform([x])[0] if x in k else -1
    )

fresh_aligned = fresh_feat.reindex(columns=X_train.columns, fill_value=0)

# Predicciones
probs = xgb_model.predict_proba(fresh_aligned)[:, 1]

print(f"Predicciones: Media {probs.mean():.2%}, Min {probs.min():.2%}, Max {probs.max():.2%}")

Predicciones: Media 53.19%, Min 0.71%, Max 82.99%


## 3. Segmentación en 5 Grupos

In [264]:
# ── Segmentación ──────────────────────────────────────────────
def segmento_riesgo(prob):
    if prob >= 0.80: return 'MUY_ALTO'
    if prob >= 0.60: return 'ALTO'
    if prob >= 0.40: return 'MEDIO'
    if prob >= 0.20: return 'BAJO'
    return 'MUY_BAJO'

clientes = df_fresh[['CODE', 'Customer_ID', 'Modelo', 'ZONA', 'PVP', 'Edad', 'Revisiones']].copy()
clientes['prob_churn'] = probs
clientes['segmento'] = clientes['prob_churn'].apply(segmento_riesgo)

print("\nDistribución por segmento:")
for seg in ['MUY_BAJO', 'BAJO', 'MEDIO', 'ALTO', 'MUY_ALTO']:
    n = (clientes['segmento'] == seg).sum()
    if n > 0:
        prob_media = clientes[clientes['segmento'] == seg]['prob_churn'].mean()
        print(f"  {seg:10s}: {n:5,} clientes ({n/len(clientes)*100:5.1f}%) - Prob media: {prob_media:.2%}")


Distribución por segmento:
  MUY_BAJO  : 1,450 clientes ( 14.5%) - Prob media: 6.80%
  BAJO      :    31 clientes (  0.3%) - Prob media: 27.21%
  MEDIO     : 3,916 clientes ( 39.2%) - Prob media: 53.84%
  ALTO      : 4,343 clientes ( 43.4%) - Prob media: 66.60%
  MUY_ALTO  :   260 clientes (  2.6%) - Prob media: 81.12%


## 4. Cálculo de Costes y Beneficios (Fórmula del Profesor)

In [265]:
# ── Fusionar con Costes.csv (solo Mantenimiento_medio) ───────
clientes = clientes.merge(costes[['Modelo', 'Mantenimiento_medio']], on='Modelo', how='left')

# ── Fórmula del Profesor ──────────────────────────────────────
# C(n) = BASE × (1 + α)^n
#   BASE = Mantenimiento_medio
#   α    = 7% para modelos A y B, 10% para el resto
#   n    = número de la PRÓXIMA revisión = Revisiones + 1
#
# Costes fijos totales = 38% del ingreso  (1% mkt + 30% CF + 7% comisión)
# Beneficio neto       = C(n) × 0.62

def calcular_ingreso(row):
    base  = row['Mantenimiento_medio']
    alpha = 0.07 if row['Modelo'] in ['A', 'B'] else 0.10
    n     = row['Revisiones'] + 1          # próxima revisión
    return base * ((1 + alpha) ** n)

clientes['C_n']            = clientes.apply(calcular_ingreso, axis=1)
clientes['Margen_Bruto']   = clientes['C_n'] * 0.62   # beneficio neto por revisión
clientes['Coste_Marketing']= clientes['C_n'] * 0.01   # 1% del ingreso

print("\n===== ANÁLISIS POR MODELO (FÓRMULA CORREGIDA) =====\n")
resumen = clientes.groupby('Modelo').agg(
    BASE   = ('Mantenimiento_medio', 'first'),
    N      = ('C_n', 'count'),
    C_n    = ('C_n', 'mean'),
    Margen_Bruto = ('Margen_Bruto', 'mean'),
    Coste_Marketing = ('Coste_Marketing', 'mean')
).round(2)
print(resumen.to_string())

print("\n💡 INTERPRETACIÓN:")
print("   C(n) = ingreso esperado de la próxima revisión")
print("   Beneficio neto = C(n) × 0.62  (después de mkt, CF y comisión)")
print(f"\n   Ingreso medio por revisión: {clientes['C_n'].mean():,.0f}€")
print(f"   Beneficio neto medio:       {clientes['Margen_Bruto'].mean():,.0f}€")


===== ANÁLISIS POR MODELO (FÓRMULA CORREGIDA) =====

         BASE     N    C_n  Margen_Bruto  Coste_Marketing
Modelo                                                   
A      250.00  1277 267.50        165.85             2.68
B      263.00  2779 281.41        174.47             2.81
C      276.00   705 303.60        188.23             3.04
D      290.00  1547 319.00        197.78             3.19
E      305.00   118 335.50        208.01             3.36
F      320.00     2 352.00        218.24             3.52
G      336.00     7 369.60        229.15             3.70
H      353.00   473 388.30        240.75             3.88
I      371.00  2625 408.10        253.02             4.08
J      390.00   451 429.00        265.98             4.29
K      410.00    16 451.00        279.62             4.51

💡 INTERPRETACIÓN:
   C(n) = ingreso esperado de la próxima revisión
   Beneficio neto = C(n) × 0.62  (después de mkt, CF y comisión)

   Ingreso medio por revisión: 333€
   Beneficio neto med

## 5. Estrategias Comerciales Diferenciadas (VERSIÓN FINAL)

### Filosofía de la Campaña

**Principio clave:** Con márgenes de 500-900€, podemos ser **muy generosos** sin comprometer rentabilidad.

- **Alto riesgo (MUY_ALTO)**: Inversión moderada + lavado premium
- **Riesgo moderado-alto (ALTO)**: **Estrategia PREMIUM VIP** completa
- **Riesgo medio (MEDIO)**: Estrategia premium intermedia
- **Riesgo bajo (BAJO/MUY_BAJO)**: Estrategia estándar + **Paquetes de retención**

### Detalle de Estrategias

| Segmento | Acciones | Coste Contacto | Costes Adicionales | **Bono** | **Extra** | Response Rate |
|----------|----------|----------------|-------------------|----------|-----------|---------------|
| **MUY_ALTO** | Email + **Lavado Completo** | 0.20€ | Lavado: 30€ | **20€** | - | 10% |
| **ALTO** | Email + **Recogida** + **Lavado** + **Neumáticos** | 0.10€ | Recogida: 35€<br>Lavado: 30€<br>Neumáticos: 35€ | **100€** | - | **80%** |
| **MEDIO** | SMS + **Regalo** + **Lavado** | 0.50€ | Regalo: 6€<br>Lavado: 30€ | **50€** | - | 55% |
| **BAJO** | SMS | 0.50€ | - | **35€** | **📦 Paquete 5 rev. -15%** | 45% |
| **MUY_BAJO** | Email | 0.20€ | - | **10€** | **📦 Paquete 5 rev. -15%** | 40% |

---

**Estrategia Premium VIP (ALTO) - Máxima captación:**
- ✅ Email personalizado
- ✅ **¡Bono de 100€!**
- ✅ **Recogida del coche** a domicilio/trabajo
- ✅ **Lavado completo** interior + exterior
- ✅ **Cambio de neumáticos** gratuito

**Estrategia Premium Intermedia (MEDIO):**
- ✅ SMS personalizado
- ✅ **¡Bono de 50€!**
- ✅ **Lavado completo** interior + exterior
- ✅ Regalo de marca

**Estrategia de Retención MUY_ALTO:**
- ✅ Email personalizado
- ✅ **Bono de 20€**
- ✅ **Lavado completo** interior + exterior

**Estrategia de Lock-in (BAJO/MUY_BAJO) - Retención a largo plazo:**
- ✅ Bono inmediato (35€ o 10€)
- ✅ **Opción EXCLUSIVA:** Paquete 5 revisiones con **15% descuento**
  - Precio normal: ~3,500€
  - Precio paquete: **2,975€**
  - **Ahorro: 525€**
  - **Beneficio:** Cliente retenido 5 revisiones (~3-5 años)

In [266]:
# ── Configuración de Estrategias FINALES ──────────────────────

# Costes de contacto por segmento
COSTE_CONTACTO = {
    'MUY_ALTO': 0.20,   # Email
    'ALTO': 0.10,       # Email personalizado
    'MEDIO': 0.50,      # SMS
    'BAJO': 0.50,       # SMS
    'MUY_BAJO': 0.20    # Email
}

# Costes adicionales (solo para los que responden)
COSTE_ADICIONAL = {
    'MUY_ALTO':  0.00,               # Sin servicios adicionales
    'ALTO': 35.00 + 30.00 + 35.00,   # Recogida + Lavado + Neumáticos
    'MEDIO': 6.00 + 30.00,            # Regalo + Lavado
    'BAJO': 0.00,
    'MUY_BAJO': 0.00
}

# Response Rate por segmento
RESPONSE_RATE = {
    'MUY_ALTO': 0.10,   # 10% - Difícil recuperarlos
    'ALTO': 0.80,       # 80% - Estrategia premium VIP
    'MEDIO': 0.55,      # 55% - Estrategia premium intermedia
    'BAJO': 0.45,       # 45% - Captación moderada
    'MUY_BAJO': 0.40    # 40% - Ya están fidelizados
}

# BONOS FIJOS (en euros)
BONOS_FIJOS = {
    'MUY_ALTO': 20.00,   # Bono 20€
    'ALTO':     50.00,   # Bono 50€
    'MEDIO':    30.00,   # Bono 30€
    'BAJO':     20.00,   # Bono 20€
    'MUY_BAJO': 10.00    # Bono 10€
}

# Aplicar estrategias
clientes['Bono_€'] = clientes['segmento'].map(BONOS_FIJOS)
clientes['Descuento_€'] = clientes['Bono_€']  # Para compatibilidad
clientes['RR'] = clientes['segmento'].map(RESPONSE_RATE)
clientes['Coste_Contacto'] = clientes['segmento'].map(COSTE_CONTACTO)
clientes['Coste_Adicional'] = clientes['segmento'].map(COSTE_ADICIONAL)

# Beneficio neto (después de bono)
clientes['Beneficio_Neto'] = clientes['Margen_Bruto'] - clientes['Bono_€']

# ── Tabla resumen ─────────────────────────────────────────────
print("\n" + "="*140)
print("ESTRATEGIAS COMERCIALES FINALES - Con Paquetes de Retención")
print("="*140)

estrategias_df = pd.DataFrame({
    'Segmento': ['MUY_BAJO', 'BAJO', 'MEDIO', 'ALTO', 'MUY_ALTO'],
    'Acción': [
        'Email + Paquete 5 rev.',
        'SMS + Paquete 5 rev.',
        'SMS + Regalo + Lavado',
        'Email + Recog. + Lav. + Neum.',
        'Email + Lavado'
    ],
    'Contacto_€': [0.20, 0.50, 0.50, 0.10, 0.20],
    'Adicional_€': [0.00, 0.00, 36.00, 100.00, 0.00],
    'Bono_€': [10.00, 20.00, 30.00, 50.00, 20.00],
    'RR_%': [40, 45, 55, 80, 10],
    'Paquete': ['📦 5 rev. -15%', '📦 5 rev. -15%', '-', '-', '-'],
    'N_Clientes': [
        (clientes['segmento'] == 'MUY_BAJO').sum(),
        (clientes['segmento'] == 'BAJO').sum(),
        (clientes['segmento'] == 'MEDIO').sum(),
        (clientes['segmento'] == 'ALTO').sum(),
        (clientes['segmento'] == 'MUY_ALTO').sum()
    ]
})

print(estrategias_df.to_string(index=False))

print("\n💎 ALTO: Bono 50€ + Recogida + Lavado Completo + Cambio Neumáticos")
print("🎁 MEDIO: Bono 30€ + Regalo + Lavado Completo")
print("⚡ MUY_ALTO: Bono 20€ (solo email, sin servicios adicionales)")
print("📦 BAJO/MUY_BAJO: Bono + Opción Paquete 5 Revisiones -15% (Retención garantizada)")
print("="*140)

# ── Análisis de Paquetes (LTV) ────────────────────────────────
print("\n" + "="*140)
print("IMPACTO DE PAQUETES DE RETENCIÓN (Valor a Largo Plazo)")
print("="*140)

n_bajo     = (clientes['segmento'] == 'BAJO').sum()
n_muy_bajo = (clientes['segmento'] == 'MUY_BAJO').sum()

n_bajo_responden     = n_bajo     * RESPONSE_RATE['BAJO']
n_muy_bajo_responden = n_muy_bajo * RESPONSE_RATE['MUY_BAJO']

n_paquetes_bajo     = n_bajo_responden     * 0.50
n_paquetes_muy_bajo = n_muy_bajo_responden * 0.60

# LTV por paquete: 5 revisiones. Descuento 15% sobre el ingreso bruto (C_n).
c_n_medio              = clientes['C_n'].mean()
margen_medio           = clientes['Margen_Bruto'].mean()
ltv_paquete            = margen_medio * 5
descuento_paquete      = c_n_medio * 5 * 0.15
beneficio_neto_paquete = ltv_paquete - descuento_paquete

for seg_name, n_resp, n_paq in [
    ('BAJO',     n_bajo_responden,     n_paquetes_bajo),
    ('MUY_BAJO', n_muy_bajo_responden, n_paquetes_muy_bajo)
]:
    print(f"\n{seg_name}:")
    print(f"  Responden: {n_resp:.0f}")
    print(f"  Compran paquete: {n_paq:.0f}")
    print(f"  Beneficio total paquetes: {beneficio_neto_paquete * n_paq:,.0f}€")

total_paquetes         = n_paquetes_bajo + n_paquetes_muy_bajo
beneficio_ltv_paquetes = beneficio_neto_paquete * total_paquetes

print(f"\n💰 IMPACTO TOTAL PAQUETES:")
print(f"  LTV por paquete (5 rev.): {ltv_paquete:,.0f}€  |  Descuento 15%: {descuento_paquete:,.0f}€  |  Neto: {beneficio_neto_paquete:,.0f}€")
print(f"  Total paquetes vendidos: {total_paquetes:.0f}")
print(f"  Beneficio LTV adicional: {beneficio_ltv_paquetes:,.0f}€ (próximos 3-5 años)")
print("="*140)


ESTRATEGIAS COMERCIALES FINALES - Con Paquetes de Retención
Segmento                        Acción  Contacto_€  Adicional_€  Bono_€  RR_%       Paquete  N_Clientes
MUY_BAJO        Email + Paquete 5 rev.        0.20         0.00   10.00    40 📦 5 rev. -15%        1450
    BAJO          SMS + Paquete 5 rev.        0.50         0.00   20.00    45 📦 5 rev. -15%          31
   MEDIO         SMS + Regalo + Lavado        0.50        36.00   30.00    55             -        3916
    ALTO Email + Recog. + Lav. + Neum.        0.10       100.00   50.00    80             -        4343
MUY_ALTO                Email + Lavado        0.20         0.00   20.00    10             -         260

💎 ALTO: Bono 50€ + Recogida + Lavado Completo + Cambio Neumáticos
🎁 MEDIO: Bono 30€ + Regalo + Lavado Completo
⚡ MUY_ALTO: Bono 20€ (solo email, sin servicios adicionales)
📦 BAJO/MUY_BAJO: Bono + Opción Paquete 5 Revisiones -15% (Retención garantizada)

IMPACTO DE PAQUETES DE RETENCIÓN (Valor a Largo Plazo)

BAJO

In [267]:
# ── COMPARATIVA: Bonos Fijos vs Sistema de Descuentos Anterior ───────────────
# El sistema anterior usaba descuentos % sobre el precio. Ahora usamos bonos fijos.
# Comparamos el coste real del descuento anterior (% sobre C_n) vs el bono nuevo.

print("\n" + "="*90)
print("COMPARATIVA: Bonos Fijos vs Sistema de Descuentos Anterior (% sobre ingreso)")
print("="*90)

DESCUENTOS_ANTERIORES = {
    'MUY_ALTO': 0.05,
    'ALTO':     0.15,
    'MEDIO':    0.12,
    'BAJO':     0.08,
    'MUY_BAJO': 0.02
}

for seg in ['MUY_ALTO', 'ALTO', 'MEDIO', 'BAJO', 'MUY_BAJO']:
    seg_clientes = clientes[clientes['segmento'] == seg]
    if len(seg_clientes) == 0:
        continue

    c_n_medio         = seg_clientes['C_n'].mean()
    desc_anterior_eur = c_n_medio * DESCUENTOS_ANTERIORES[seg]
    bono_nuevo        = BONOS_FIJOS[seg]
    ahorro            = desc_anterior_eur - bono_nuevo

    print(f"\n{seg}")
    print(f"  Ingreso medio (C_n): {c_n_medio:.2f}€")
    print(f"  Descuento anterior ({DESCUENTOS_ANTERIORES[seg]*100:.0f}%): {desc_anterior_eur:.2f}€")
    print(f"  Bono fijo nuevo:     {bono_nuevo:.0f}€")
    print(f"  ➤ Ahorro: {ahorro:.2f}€ por cliente ({ahorro/desc_anterior_eur*100:.1f}% menos)")

print("\n" + "="*90)


COMPARATIVA: Bonos Fijos vs Sistema de Descuentos Anterior (% sobre ingreso)

MUY_ALTO
  Ingreso medio (C_n): 359.27€
  Descuento anterior (5%): 17.96€
  Bono fijo nuevo:     20€
  ➤ Ahorro: -2.04€ por cliente (-11.3% menos)

ALTO
  Ingreso medio (C_n): 314.17€
  Descuento anterior (15%): 47.13€
  Bono fijo nuevo:     50€
  ➤ Ahorro: -2.87€ por cliente (-6.1% menos)

MEDIO
  Ingreso medio (C_n): 350.15€
  Descuento anterior (12%): 42.02€
  Bono fijo nuevo:     30€
  ➤ Ahorro: 12.02€ por cliente (28.6% menos)

BAJO
  Ingreso medio (C_n): 395.96€
  Descuento anterior (8%): 31.68€
  Bono fijo nuevo:     20€
  ➤ Ahorro: 11.68€ por cliente (36.9% menos)

MUY_BAJO
  Ingreso medio (C_n): 336.81€
  Descuento anterior (2%): 6.74€
  Bono fijo nuevo:     10€
  ➤ Ahorro: -3.26€ por cliente (-48.5% menos)



In [268]:
# ── ROI por segmento (con estrategias diferenciadas) ──────────
# ROI = (Margen_Bruto - Costes_Campaña) / Costes_Campaña
# Costes_Campaña = Coste_Contacto + Coste_Adicional + Coste_Marketing + Descuento
# NO incluir CF

resultados_roi = []

for segmento in ['MUY_ALTO', 'ALTO', 'MEDIO', 'BAJO', 'MUY_BAJO']:
    seg = clientes[clientes['segmento'] == segmento]
    if len(seg) == 0:
        continue
    
    n_total = len(seg)
    rr = RESPONSE_RATE[segmento]
    n_responden = n_total * rr
    
    # Costes de campaña
    coste_contacto_unitario = COSTE_CONTACTO[segmento]
    coste_adicional_unitario = COSTE_ADICIONAL[segmento]
    
    # Se contacta a TODOS
    coste_contacto_total = n_total * coste_contacto_unitario
    
    # Costes adicionales solo para los que responden
    coste_adicional_total = n_responden * coste_adicional_unitario
    coste_marketing_total = n_responden * seg['Coste_Marketing'].mean()
    coste_descuentos = n_responden * seg['Descuento_€'].mean()
    
    costes_campana = (coste_contacto_total + coste_adicional_total + 
                      coste_marketing_total + coste_descuentos)
    
    # Margen bruto de los que responden
    margen_bruto_total = n_responden * seg['Margen_Bruto'].mean()
    
    # Beneficio neto
    beneficio_neto = margen_bruto_total - costes_campana
    
    # ROI
    roi = (beneficio_neto / costes_campana) if costes_campana > 0 else 0
    
    resultados_roi.append({
        'Segmento': segmento,
        'N_Clientes': n_total,
        'RR_%': int(rr * 100),
        'N_Responden': int(n_responden),
        'Coste_Contacto_€': coste_contacto_total,
        'Coste_Adicional_€': coste_adicional_total,
        'Coste_Marketing_€': coste_marketing_total,
        'Coste_Descuentos_€': coste_descuentos,
        'Costes_Total_€': costes_campana,
        'Margen_Bruto_€': margen_bruto_total,
        'Beneficio_Neto_€': beneficio_neto,
        'ROI': roi
    })

df_roi = pd.DataFrame(resultados_roi)

# ROI Global
costes_total = df_roi['Costes_Total_€'].sum()
beneficio_total = df_roi['Beneficio_Neto_€'].sum()
roi_global = (beneficio_total / costes_total) if costes_total > 0 else 0

print("\n" + "="*130)
print("ROI POR SEGMENTO (Estrategias Finales)")
print("="*130)
print(df_roi[['Segmento', 'N_Clientes', 'RR_%', 'N_Responden', 
              'Costes_Total_€', 'Margen_Bruto_€', 'Beneficio_Neto_€', 'ROI']].to_string(index=False))

print("\n" + "="*130)
print(f"{'ROI GLOBAL:':40s} {roi_global:.4f}")
print(f"{'Inversión Total Campaña:':40s} {costes_total:,.2f}€")
print(f"{'Beneficio Neto Total:':40s} {beneficio_total:,.2f}€")
print(f"{'Clientes que responderán (estimado):':40s} {int(df_roi['N_Responden'].sum()):,}")
print("="*130)


ROI POR SEGMENTO (Estrategias Finales)
Segmento  N_Clientes  RR_%  N_Responden  Costes_Total_€  Margen_Bruto_€  Beneficio_Neto_€   ROI
MUY_ALTO         260    10           26          665.41        5,791.41          5,126.00  7.70
    ALTO        4343    80         3474      532,509.79      676,760.56        144,250.77  0.27
   MEDIO        3916    55         2153      151,650.27      467,571.01        315,920.74  2.08
    BAJO          31    45           13          349.74        3,424.70          3,074.96  8.79
MUY_BAJO        1450    40          580        8,043.49      121,116.56        113,073.07 14.06

ROI GLOBAL:                              0.8388
Inversión Total Campaña:                 693,218.70€
Beneficio Neto Total:                    581,445.53€
Clientes que responderán (estimado):     6,246


In [269]:
# ── Desglose detallado de ROI por cliente ────────────────────
print("\n" + "="*150)
print("ANÁLISIS DETALLADO: ROI POR CLIENTE (Muestra de 20 clientes)")
print("="*150)

muestra_clientes = []
for seg in ['MUY_ALTO', 'ALTO', 'MEDIO', 'BAJO', 'MUY_BAJO']:
    seg_clientes = clientes[clientes['segmento'] == seg]
    n = min(4, len(seg_clientes))
    if n > 0:
        muestra_clientes.append(seg_clientes.sample(n=n, random_state=42))

muestra = pd.concat(muestra_clientes)

muestra_analisis = muestra[['CODE', 'Modelo', 'segmento', 'C_n', 'Margen_Bruto',
                             'Coste_Contacto', 'Coste_Adicional', 'Coste_Marketing',
                             'Bono_€', 'RR']].copy()

muestra_analisis['Total_Costes_Campaña'] = (
    muestra_analisis['Coste_Contacto'] +
    muestra_analisis['Coste_Adicional'] +
    muestra_analisis['Coste_Marketing'] +
    muestra_analisis['Bono_€']
)

muestra_analisis['Beneficio_Si_Responde'] = (
    muestra_analisis['Margen_Bruto'] - muestra_analisis['Total_Costes_Campaña']
)

muestra_analisis['ROI_Si_Responde'] = (
    muestra_analisis['Beneficio_Si_Responde'] / muestra_analisis['Total_Costes_Campaña']
).replace([np.inf, -np.inf], 0)

muestra_analisis['Beneficio_Esperado'] = (
    muestra_analisis['Beneficio_Si_Responde'] * muestra_analisis['RR'] -
    muestra_analisis['Coste_Contacto']
)

tabla_display = muestra_analisis[[
    'CODE', 'Modelo', 'segmento', 'C_n', 'Margen_Bruto',
    'Coste_Contacto', 'Coste_Adicional', 'Bono_€', 'Total_Costes_Campaña',
    'Beneficio_Si_Responde', 'ROI_Si_Responde', 'RR', 'Beneficio_Esperado'
]].copy()

tabla_display.columns = [
    'Cliente', 'Mod', 'Seg', 'Ingreso€', 'Marg_Bruto€',
    'Contacto€', 'Adicional€', 'Bono€', 'Total_Costes€',
    'Benef_Si_Resp€', 'ROI_Resp', 'RR', 'Benef_Esper€'
]
tabla_display['RR'] = (tabla_display['RR'] * 100).astype(int)

print(tabla_display.to_string(index=False))

print("\n" + "="*150)
print("RESUMEN POR SEGMENTO (de la muestra)")
print("="*150)

resumen_muestra = tabla_display.groupby('Seg').agg({
    'Ingreso€': 'mean', 'Marg_Bruto€': 'mean', 'Total_Costes€': 'mean',
    'Benef_Si_Resp€': 'mean', 'ROI_Resp': 'mean', 'Benef_Esper€': 'mean'
}).round(2)
print(resumen_muestra.to_string())
print("="*150)


ANÁLISIS DETALLADO: ROI POR CLIENTE (Muestra de 20 clientes)
  Cliente Mod      Seg  Ingreso€  Marg_Bruto€  Contacto€  Adicional€  Bono€  Total_Costes€  Benef_Si_Resp€  ROI_Resp  RR  Benef_Esper€
SYN001297   H MUY_ALTO    388.30       240.75       0.20        0.00  20.00          24.08          216.66      9.00  10         21.47
SYN007285   H MUY_ALTO    388.30       240.75       0.20        0.00  20.00          24.08          216.66      9.00  10         21.47
SYN008866   I MUY_ALTO    408.10       253.02       0.20        0.00  20.00          24.28          228.74      9.42  10         22.67
SYN007486   I MUY_ALTO    408.10       253.02       0.20        0.00  20.00          24.28          228.74      9.42  10         22.67
SYN005779   A     ALTO    267.50       165.85       0.10      100.00  50.00         152.77           13.08      0.09  80         10.36
SYN000756   B     ALTO    281.41       174.47       0.10      100.00  50.00         152.91           21.56      0.14  80        

## 6. Análisis Detallado por Cliente (Desglose de ROI)

## 7. Visualizaciones - Análisis de Resultados

### Gráficos profesionales para entender el impacto de la campaña

In [270]:
# ══════════════════════════════════════════════════════════════
# GRÁFICO 5: Impacto Global de la Campaña
# ══════════════════════════════════════════════════════════════

# Preparar datos para waterfall
categorias = ['Inversión\nTotal', 'Beneficio\nALTO', 'Beneficio\nMEDIO', 
              'Beneficio\nBAJO', 'Beneficio\nMUY_BAJO', 'Beneficio\nMUY_ALTO', 'Beneficio\nNeto Total']

valores = [
    -costes_total,  # Inversión (negativa)
    df_roi[df_roi['Segmento'] == 'ALTO']['Beneficio_Neto_€'].iloc[0] if len(df_roi[df_roi['Segmento'] == 'ALTO']) > 0 else 0,
    df_roi[df_roi['Segmento'] == 'MEDIO']['Beneficio_Neto_€'].iloc[0] if len(df_roi[df_roi['Segmento'] == 'MEDIO']) > 0 else 0,
    df_roi[df_roi['Segmento'] == 'BAJO']['Beneficio_Neto_€'].iloc[0] if len(df_roi[df_roi['Segmento'] == 'BAJO']) > 0 else 0,
    df_roi[df_roi['Segmento'] == 'MUY_BAJO']['Beneficio_Neto_€'].iloc[0] if len(df_roi[df_roi['Segmento'] == 'MUY_BAJO']) > 0 else 0,
    df_roi[df_roi['Segmento'] == 'MUY_ALTO']['Beneficio_Neto_€'].iloc[0] if len(df_roi[df_roi['Segmento'] == 'MUY_ALTO']) > 0 else 0,
    beneficio_total
]

medidas = ['absolute', 'relative', 'relative', 'relative', 'relative', 'relative', 'total']

fig5 = go.Figure(go.Waterfall(
    x=categorias,
    y=valores,
    measure=medidas,
    text=[f"{v:,.0f}€" for v in valores],
    textposition="outside",
    connector={"line": {"color": "rgb(63, 63, 63)"}},
    decreasing={"marker": {"color": "#e74c3c"}},
    increasing={"marker": {"color": "#27ae60"}},
    totals={"marker": {"color": "#3498db"}}
))

fig5.update_layout(
    title=dict(
        text='Impacto Global de la Campaña - Waterfall',
        font=dict(size=20, color='white')
    ),
    yaxis_title='Euros (€)',
    template='plotly_dark',
    height=500,
    showlegend=False
)

fig5.show()

print("📊 Interpretación: Empezamos con la inversión (rojo) y cada segmento aporta beneficio (verde).")
print(f"   El resultado final (azul) es {beneficio_total:,.0f}€ de beneficio neto.")

📊 Interpretación: Empezamos con la inversión (rojo) y cada segmento aporta beneficio (verde).
   El resultado final (azul) es 581,446€ de beneficio neto.


In [271]:
# ══════════════════════════════════════════════════════════════
# GRÁFICO 4: Desglose de Costes por Segmento
# ══════════════════════════════════════════════════════════════

fig4 = go.Figure()

fig4.add_trace(go.Bar(
    name='Contacto',
    x=df_roi['Segmento'],
    y=df_roi['Coste_Contacto_€'],
    marker_color='#95a5a6',
    text=[f"{v:.0f}€" for v in df_roi['Coste_Contacto_€']],
    textposition='inside'
))

fig4.add_trace(go.Bar(
    name='Servicios (Recogida/Lavado/Regalo)',
    x=df_roi['Segmento'],
    y=df_roi['Coste_Adicional_€'],
    marker_color='#3498db',
    text=[f"{v:.0f}€" for v in df_roi['Coste_Adicional_€']],
    textposition='inside'
))

fig4.add_trace(go.Bar(
    name='Marketing (1% CF)',
    x=df_roi['Segmento'],
    y=df_roi['Coste_Marketing_€'],
    marker_color='#9b59b6',
    text=[f"{v:.0f}€" for v in df_roi['Coste_Marketing_€']],
    textposition='inside'
))

fig4.add_trace(go.Bar(
    name='Bonos',
    x=df_roi['Segmento'],
    y=df_roi['Coste_Descuentos_€'],
    marker_color='#e67e22',
    text=[f"{v:,.0f}€" for v in df_roi['Coste_Descuentos_€']],
    textposition='inside'
))

fig4.update_layout(
    title=dict(
        text='Desglose de Costes de Campaña por Segmento',
        font=dict(size=20, color='white')
    ),
    xaxis_title='Segmento',
    yaxis_title='Euros (€)',
    barmode='stack',
    template='plotly_dark',
    height=500,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig4.show()

print("📊 Interpretación: ALTO tiene costes altos por la estrategia premium (recogida + lavado + neumáticos).")
print("   Pero el ROI sigue siendo excelente gracias a los márgenes de 500-900€ por cliente.")

📊 Interpretación: ALTO tiene costes altos por la estrategia premium (recogida + lavado + neumáticos).
   Pero el ROI sigue siendo excelente gracias a los márgenes de 500-900€ por cliente.


In [272]:
# ══════════════════════════════════════════════════════════════
# GRÁFICO 3: Distribución de Clientes y Captación
# ══════════════════════════════════════════════════════════════

fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Distribución de Clientes por Segmento', 'Tasa de Captación (Response Rate)'),
    specs=[[{'type': 'pie'}, {'type': 'bar'}]]
)

# Pie chart - Distribución de clientes
fig3.add_trace(
    go.Pie(
        labels=df_roi['Segmento'],
        values=df_roi['N_Clientes'],
        marker=dict(colors=['#e74c3c', '#e67e22', '#f39c12', '#27ae60', '#3498db']),
        textinfo='label+percent',
        textfont=dict(size=12),
        hovertemplate='<b>%{label}</b><br>%{value:,} clientes<br>%{percent}<extra></extra>'
    ),
    row=1, col=1
)

# Bar chart - Response Rate
fig3.add_trace(
    go.Bar(
        x=df_roi['Segmento'],
        y=df_roi['RR_%'],
        marker_color='#3498db',
        text=[f"{v}%" for v in df_roi['RR_%']],
        textposition='outside',
        textfont=dict(size=12),
        showlegend=False
    ),
    row=1, col=2
)

fig3.update_xaxes(title_text="Segmento", row=1, col=2)
fig3.update_yaxes(title_text="Response Rate (%)", row=1, col=2)

fig3.update_layout(
    title=dict(
        text='Distribución y Captación de Clientes',
        font=dict(size=20, color='white')
    ),
    template='plotly_dark',
    height=500,
    showlegend=False
)

fig3.show()

print("📊 Interpretación: A la izquierda vemos cuántos clientes hay en cada segmento.")
print("   A la derecha, qué % esperamos captar con cada estrategia (ALTO tiene 80% por estrategia premium).")

📊 Interpretación: A la izquierda vemos cuántos clientes hay en cada segmento.
   A la derecha, qué % esperamos captar con cada estrategia (ALTO tiene 80% por estrategia premium).


In [273]:
# ══════════════════════════════════════════════════════════════
# GRÁFICO 2: Inversión vs Beneficio por Segmento
# ══════════════════════════════════════════════════════════════

fig2 = go.Figure()

fig2.add_trace(go.Bar(
    name='Inversión',
    x=df_roi['Segmento'],
    y=df_roi['Costes_Total_€'],
    marker_color='#e74c3c',
    text=[f"{v:,.0f}€" for v in df_roi['Costes_Total_€']],
    textposition='outside',
    textfont=dict(size=12)
))

fig2.add_trace(go.Bar(
    name='Beneficio',
    x=df_roi['Segmento'],
    y=df_roi['Beneficio_Neto_€'],
    marker_color='#27ae60',
    text=[f"{v:,.0f}€" for v in df_roi['Beneficio_Neto_€']],
    textposition='outside',
    textfont=dict(size=12)
))

fig2.update_layout(
    title=dict(
        text='Inversión vs Beneficio por Segmento',
        font=dict(size=20, color='white')
    ),
    xaxis_title='Segmento',
    yaxis_title='Euros (€)',
    template='plotly_dark',
    height=500,
    barmode='group',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig2.show()

print("📊 Interpretación: Las barras verdes (beneficio) deben ser mayores que las rojas (inversión) para ser rentables.")

📊 Interpretación: Las barras verdes (beneficio) deben ser mayores que las rojas (inversión) para ser rentables.


In [274]:
# ══════════════════════════════════════════════════════════════
# GRÁFICO 1: ROI por Segmento
# ══════════════════════════════════════════════════════════════

fig1 = go.Figure()

# Colores según ROI
colores = ['#27ae60' if r >= 3 else '#f39c12' if r >= 1 else '#e74c3c' for r in df_roi['ROI']]

fig1.add_trace(go.Bar(
    x=df_roi['Segmento'],
    y=df_roi['ROI'],
    marker_color=colores,
    text=[f"{v:.2f}" for v in df_roi['ROI']],
    textposition='outside',
    textfont=dict(size=14, color='white'),
    hovertemplate='<b>%{x}</b><br>ROI: %{y:.2f}<br>(%{y:.0%})<extra></extra>'
))

fig1.add_hline(y=1, line_dash='dash', line_color='white', line_width=2,
               annotation_text='Punto de equilibrio (ROI=1)', annotation_position='right')

fig1.update_layout(
    title=dict(
        text='ROI por Segmento - Retorno de la Inversión',
        font=dict(size=20, color='white')
    ),
    xaxis_title='Segmento de Clientes',
    yaxis_title='ROI (Beneficio / Inversión)',
    template='plotly_dark',
    height=500,
    showlegend=False,
    font=dict(size=12)
)

fig1.show()

print("📊 Interpretación: Un ROI de 5.0 significa que por cada 1€ invertido, ganamos 5€ de beneficio.")

📊 Interpretación: Un ROI de 5.0 significa que por cada 1€ invertido, ganamos 5€ de beneficio.


---

# 📊 Dashboard Ejecutivo de Retención

## Visualizaciones avanzadas para presentación ejecutiva

**Objetivo:** Comunicar de forma clara y profesional los resultados de la estrategia de retención mediante gráficos ejecutivos de alto impacto.

In [275]:
# ══════════════════════════════════════════════════════════════════════════════
# 1️⃣ BUBBLE CHART: Riesgo de Churn vs Rentabilidad por Segmento
# ══════════════════════════════════════════════════════════════════════════════

# Preparar datos agregados por segmento
bubble_data = []

for _, row in df_roi.iterrows():
    segmento = row['Segmento']
    
    # Datos del segmento
    seg_clientes = clientes[clientes['segmento'] == segmento]
    
    if len(seg_clientes) > 0:
        prob_churn_media = seg_clientes['prob_churn'].mean()
        margen_neto_medio = row['Beneficio_Neto_€'] / row['N_Responden'] if row['N_Responden'] > 0 else 0
        n_clientes = row['N_Clientes']
        roi = row['ROI']
        
        bubble_data.append({
            'Segmento': segmento,
            'Prob_Churn_%': prob_churn_media * 100,
            'Margen_Neto_€': margen_neto_medio,
            'N_Clientes': n_clientes,
            'ROI': roi
        })

df_bubble = pd.DataFrame(bubble_data)

# Colores por segmento (degradado de verde a rojo)
color_map = {
    'MUY_BAJO': '#2ecc71',  # Verde
    'BAJO': '#f39c12',      # Naranja
    'MEDIO': '#e67e22',     # Naranja oscuro
    'ALTO': '#e74c3c',      # Rojo
    'MUY_ALTO': '#c0392b'   # Rojo oscuro
}

# Crear bubble chart
fig_bubble = go.Figure()

for _, row in df_bubble.iterrows():
    fig_bubble.add_trace(go.Scatter(
        x=[row['Prob_Churn_%']],
        y=[row['Margen_Neto_€']],
        mode='markers+text',
        marker=dict(
            size=row['N_Clientes'] / 30,  # Escalar tamaño
            color=color_map[row['Segmento']],
            line=dict(width=2, color='white'),
            opacity=0.7
        ),
        text=row['Segmento'],
        textposition='middle center',
        textfont=dict(size=11, color='white', family='Arial Black'),
        name=row['Segmento'],
        hovertemplate=(
            '<b>%{text}</b><br>'
            'Prob. Churn: %{x:.1f}%<br>'
            'Margen Neto: %{y:,.0f}€<br>'
            f"N° Clientes: {row['N_Clientes']:,}<br>"
            f"ROI: {row['ROI']:.2f}"
            '<extra></extra>'
        )
    ))

# Añadir líneas de referencia
fig_bubble.add_hline(
    y=df_bubble['Margen_Neto_€'].mean(), 
    line_dash='dash', 
    line_color='gray', 
    line_width=1,
    annotation_text='Margen Neto Promedio',
    annotation_position='right'
)

fig_bubble.add_vline(
    x=df_bubble['Prob_Churn_%'].mean(), 
    line_dash='dash', 
    line_color='gray', 
    line_width=1,
    annotation_text='Prob. Churn Promedio',
    annotation_position='top'
)

fig_bubble.update_layout(
    title=dict(
        text='<b>Análisis de Segmentos: Riesgo vs Rentabilidad</b><br><sub>Tamaño de burbuja = N° de clientes</sub>',
        font=dict(size=18, color='#2c3e50', family='Arial')
    ),
    xaxis=dict(
        title='<b>Probabilidad de Churn Media (%)</b>',
        showgrid=True,
        gridcolor='#ecf0f1',
        zeroline=False,
        titlefont=dict(size=13, color='#34495e')
    ),
    yaxis=dict(
        title='<b>Margen Neto por Cliente Recuperado (€)</b>',
        showgrid=True,
        gridcolor='#ecf0f1',
        zeroline=False,
        titlefont=dict(size=13, color='#34495e')
    ),
    template='plotly_white',
    height=550,
    showlegend=True,
    legend=dict(
        title='<b>Segmentos</b>',
        orientation='v',
        yanchor='top',
        y=1,
        xanchor='left',
        x=1.02,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='#bdc3c7',
        borderwidth=1
    ),
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig_bubble.show()

print('\n📊 INTERPRETACIÓN DEL BUBBLE CHART:')
print('   • Eje X (izq → der): Mayor probabilidad de churn (mayor riesgo)')
print('   • Eje Y (abajo → arriba): Mayor margen neto por cliente (mayor rentabilidad)')
print('   • Tamaño burbuja: Mayor población de clientes')
print('   • COLOR: Verde (bajo riesgo) → Rojo (alto riesgo)')
print('\n💡 INSIGHT CLAVE:')
print(f"   El segmento ALTO (rojo, {df_bubble[df_bubble['Segmento']=='ALTO']['N_Clientes'].values[0]:,} clientes) tiene:")
print(f"   - Alta probabilidad de churn ({df_bubble[df_bubble['Segmento']=='ALTO']['Prob_Churn_%'].values[0]:.1f}%)")
print(f"   - Margen neto rentable ({df_bubble[df_bubble['Segmento']=='ALTO']['Margen_Neto_€'].values[0]:,.0f}€ por cliente)")
print('   - Estrategia premium agresiva JUSTIFICADA por volumen y rentabilidad')


📊 INTERPRETACIÓN DEL BUBBLE CHART:
   • Eje X (izq → der): Mayor probabilidad de churn (mayor riesgo)
   • Eje Y (abajo → arriba): Mayor margen neto por cliente (mayor rentabilidad)
   • Tamaño burbuja: Mayor población de clientes
   • COLOR: Verde (bajo riesgo) → Rojo (alto riesgo)

💡 INSIGHT CLAVE:
   El segmento ALTO (rojo, 4,343 clientes) tiene:
   - Alta probabilidad de churn (66.6%)
   - Margen neto rentable (42€ por cliente)
   - Estrategia premium agresiva JUSTIFICADA por volumen y rentabilidad


In [276]:
# ══════════════════════════════════════════════════════════════════════════════
# 2️⃣ STACKED BAR 100%: Composición de cada Euro de Ingreso
# ══════════════════════════════════════════════════════════════════════════════

# Preparar datos de composición
composicion_data = []

for _, row in df_roi.iterrows():
    segmento = row['Segmento']
    
    # Calcular promedios por cliente que responde
    n_responden = row['N_Responden']
    
    if n_responden > 0:
        # Por cliente que responde
        margen_bruto_unitario = row['Margen_Bruto_€'] / n_responden
        
        # Coste Fijo promedio (30% del ingreso C_n, segun estructura de costes del profesor)
        seg_clientes = clientes[clientes['segmento'] == segmento]
        cf_unitario = seg_clientes['C_n'].mean() * 0.30
        
        # Costes de campaña por cliente que responde
        coste_campana_unitario = (
            row['Coste_Adicional_€'] / n_responden +
            row['Coste_Marketing_€'] / n_responden +
            row['Coste_Descuentos_€'] / n_responden
        )
        
        # Beneficio neto unitario
        beneficio_neto_unitario = row['Beneficio_Neto_€'] / n_responden
        
        # Total (debería ser igual al precio)
        total = cf_unitario + coste_campana_unitario + beneficio_neto_unitario
        
        # Calcular porcentajes
        composicion_data.append({
            'Segmento': segmento,
            'CF_%': (cf_unitario / total) * 100,
            'Campaña_%': (coste_campana_unitario / total) * 100,
            'Beneficio_%': (beneficio_neto_unitario / total) * 100,
            'CF_€': cf_unitario,
            'Campaña_€': coste_campana_unitario,
            'Beneficio_€': beneficio_neto_unitario
        })

df_comp = pd.DataFrame(composicion_data)

# Ordenar por ROI descendente
orden_segmentos = df_roi.sort_values('ROI', ascending=False)['Segmento'].tolist()
df_comp['Segmento'] = pd.Categorical(df_comp['Segmento'], categories=orden_segmentos, ordered=True)
df_comp = df_comp.sort_values('Segmento')

# Crear stacked bar
fig_stack = go.Figure()

# Coste Fijo (gris)
fig_stack.add_trace(go.Bar(
    name='Coste Fijo (CF)',
    x=df_comp['Segmento'],
    y=df_comp['CF_%'],
    marker_color='#95a5a6',
    text=[f"{v:.1f}%<br>{e:.0f}€" for v, e in zip(df_comp['CF_%'], df_comp['CF_€'])],
    textposition='inside',
    textfont=dict(size=11, color='white'),
    hovertemplate='<b>Coste Fijo</b><br>%{y:.1f}%<br>%{text}<extra></extra>'
))

# Gastos de Campaña (naranja)
fig_stack.add_trace(go.Bar(
    name='Gastos Campaña + Marketing',
    x=df_comp['Segmento'],
    y=df_comp['Campaña_%'],
    marker_color='#e67e22',
    text=[f"{v:.1f}%<br>{e:.0f}€" for v, e in zip(df_comp['Campaña_%'], df_comp['Campaña_€'])],
    textposition='inside',
    textfont=dict(size=11, color='white'),
    hovertemplate='<b>Gastos Campaña</b><br>%{y:.1f}%<br>%{text}<extra></extra>'
))

# Beneficio Neto (verde)
fig_stack.add_trace(go.Bar(
    name='Beneficio Neto',
    x=df_comp['Segmento'],
    y=df_comp['Beneficio_%'],
    marker_color='#27ae60',
    text=[f"{v:.1f}%<br>{e:.0f}€" for v, e in zip(df_comp['Beneficio_%'], df_comp['Beneficio_€'])],
    textposition='inside',
    textfont=dict(size=11, color='white'),
    hovertemplate='<b>Beneficio Neto</b><br>%{y:.1f}%<br>%{text}<extra></extra>'
))

fig_stack.update_layout(
    title=dict(
        text='<b>Composición de cada Euro de Ingreso por Segmento</b><br><sub>Ordenado por ROI descendente (más rentable a la izquierda)</sub>',
        font=dict(size=18, color='#2c3e50', family='Arial')
    ),
    xaxis=dict(
        title='<b>Segmento de Cliente</b>',
        titlefont=dict(size=13, color='#34495e')
    ),
    yaxis=dict(
        title='<b>Composición del Ingreso (%)</b>',
        titlefont=dict(size=13, color='#34495e'),
        range=[0, 100]
    ),
    barmode='stack',
    template='plotly_white',
    height=550,
    legend=dict(
        title='<b>Componentes</b>',
        orientation='v',
        yanchor='top',
        y=1,
        xanchor='left',
        x=1.02,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='#bdc3c7',
        borderwidth=1
    ),
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig_stack.show()

print('\n📊 INTERPRETACIÓN DEL STACKED BAR:')
print('   Cada barra representa el 100% del precio que paga el cliente.')
print('   Se descompone en:')
print('   • GRIS: Coste Fijo (CF) - Piezas y mano de obra del mantenimiento')
print('   • NARANJA: Gastos Campaña + Marketing - Inversión en retención')
print('   • VERDE: Beneficio Neto - Rentabilidad real del concesionario')
print('\n💡 INSIGHT CLAVE:')
mejor_seg = df_comp.iloc[0]['Segmento']
mejor_benef = df_comp.iloc[0]['Beneficio_%']
print(f"   El segmento {mejor_seg} es el más rentable: {mejor_benef:.1f}% de beneficio neto")
print(f"   Por cada 100€ facturados, ganamos {mejor_benef:.0f}€ de beneficio puro.")


📊 INTERPRETACIÓN DEL STACKED BAR:
   Cada barra representa el 100% del precio que paga el cliente.
   Se descompone en:
   • GRIS: Coste Fijo (CF) - Piezas y mano de obra del mantenimiento
   • NARANJA: Gastos Campaña + Marketing - Inversión en retención
   • VERDE: Beneficio Neto - Rentabilidad real del concesionario

💡 INSIGHT CLAVE:
   El segmento MUY_BAJO es el más rentable: 63.0% de beneficio neto
   Por cada 100€ facturados, ganamos 63€ de beneficio puro.


In [277]:
import plotly.graph_objects as go

# ══════════════════════════════════════════════════════════════════════════════
# 1️⃣ GAUGE CHART: El indicador clave de éxito (ROI)
# ══════════════════════════════════════════════════════════════════════════════

fig_gauge = go.Figure(go.Indicator(
    mode = "gauge+number",
    value = roi_global,
    domain = {'x': [0, 1], 'y': [0, 1]},
    title = {
        'text': "<b>ROI GLOBAL DE LA ESTRATEGIA</b><br><span style='font-size:0.8em;color:gray'>Retorno por cada 1€ invertido</span>",
        'font': {'size': 24, 'family': 'Arial Black'}
    },
    number = {
        'font': {'size': 70, 'color': "#27ae60", 'family': 'Arial Black'}, 
        'suffix': "x", # Para que salga como 5.00x
        'valueformat': '.2f'
    },
    gauge = {
        'axis': {'range': [0, 6], 'tickwidth': 1, 'tickcolor': "#2c3e50"},
        'bar': {'color': "#27ae60", 'thickness': 0.8}, # El color de la aguja/barra
        'bgcolor': "white",
        'borderwidth': 1,
        'bordercolor': "#bdc3c7",
        'steps': [
            {'range': [0, 1], 'color': '#ffcccc'}, # Zona Roja: Pérdidas
            {'range': [1, 2], 'color': '#ffe5d9'}, # Zona Naranja: Recuperación
            {'range': [2, 4], 'color': '#fff3cd'}, # Zona Amarilla: Rentable
            {'range': [4, 6], 'color': '#d4edda'}  # Zona Verde: Excelente
        ],
        'threshold': {
            'line': {'color': "black", 'width': 5},
            'thickness': 0.75,
            'value': 3.0 # La marca del objetivo mínimo
        }
    }
))

# Ajustes de diseño para que se vea impecable
fig_gauge.update_layout(
    height=500,
    template='plotly_white',
    margin={'t': 100, 'b': 50, 'l': 50, 'r': 50},
    font=dict(family="Arial", color="#34495e")
)

# Añadimos un cuadro con el resumen de dinero debajo (opcional, pero muy pro)
fig_gauge.add_annotation(
    text=f"<b>Inversión Total:</b> {costes_total:,.0f}€  |  <b>Beneficio Neto:</b> {beneficio_total:,.0f}€",
    xref="paper", yref="paper",
    x=0.5, y=-0.05, showarrow=False,
    font=dict(size=16, color="#2c3e50"),
    bordercolor="#27ae60", borderpad=8, bgcolor="#f8f9fa", borderwidth=2
)

fig_gauge.show()

---

# Parte 3 (Opcional): Estrategia Basada en CLTV

## Customer Lifetime Value como criterio de priorización

Hasta ahora la acción comercial se ha basado **únicamente en la probabilidad de churn**: a mayor riesgo, mayor inversión en retención.

Sin embargo, este enfoque ignora el **valor económico del cliente**. No tiene sentido invertir la misma cantidad en retener a un cliente con margen de 500€ que a uno con margen de 8.000€, aunque ambos tengan el mismo riesgo de churn.

### Fórmula del CLTV utilizada

Usando la distribución geométrica, el número esperado de revisiones futuras antes de que un cliente haga churn es `(1 - p) / p`, donde `p` es la probabilidad de churn. Por tanto:

```
CLTV = Margen_Bruto × (1 - prob_churn) / prob_churn
```

- Un cliente con churn 10% y margen 600€ → CLTV = 600 × 9 = **5.400€**
- Un cliente con churn 70% y margen 600€ → CLTV = 600 × 0.43 = **257€**

### Lógica de la nueva estrategia (2D: Churn × CLTV)

| | CLTV Bajo (≤P33) | CLTV Medio (P33-P66) | CLTV Alto (>P66) |
|---|---|---|---|
| **Churn MUY_ALTO** | Email básico | Email + Lavado | Email + Lavado + Bono EXTRA |
| **Churn ALTO** | SMS + Bono reducido | Email + Pack Premium | Email + Pack Premium VIP |
| **Churn MEDIO** | SMS básico | SMS + Lavado | SMS + Lavado + Bono EXTRA |
| **Churn BAJO** | Email básico | SMS + Bono | SMS + Paquete 5 rev. |
| **Churn MUY_BAJO** | Sin acción | Email + Bono | Email + Paquete 5 rev. Premium |

In [278]:
# ══════════════════════════════════════════════════════════════════════════════
# PARTE 3: Cálculo del CLTV (Fórmula del Profesor)
# ══════════════════════════════════════════════════════════════════════════════
#
# CLTV = Σ(k=1..N) [ C(n+k) × 0.62 × (1 - p_churn)^k ]
#
# Donde:
#   C(n+k) = BASE × (1+α)^(n+k)   → ingreso de la revisión futura k
#   0.62                           → beneficio neto (100% - 38% costes)
#   (1 - p_churn)^k                → probabilidad de que el cliente siga activo
#   n = Revisiones actuales, N = 20 revisiones futuras máx.
#
# Forma cerrada: BASE × 0.62 × (1+α)^n × q×(1-q^N)/(1-q)
# donde q = (1+α)×(1-p_churn)
# ──────────────────────────────────────────────────────────────────────────────

N_FUTURAS = 20

clientes_cltv = clientes.copy()

def calcular_cltv(row):
    base  = row['Mantenimiento_medio']
    alpha = 0.07 if row['Modelo'] in ['A', 'B'] else 0.10
    n     = row['Revisiones']
    p     = min(row['prob_churn'], 0.999)   # evitar p=1
    q     = (1 + alpha) * (1 - p)
    if abs(q - 1) < 1e-9:
        total = N_FUTURAS
    else:
        total = q * (1 - q**N_FUTURAS) / (1 - q)
    return base * ((1 + alpha)**n) * 0.62 * total

clientes_cltv['CLTV'] = clientes_cltv.apply(calcular_cltv, axis=1)

# Clasificación en 3 niveles por percentil
p33 = clientes_cltv['CLTV'].quantile(0.33)
p66 = clientes_cltv['CLTV'].quantile(0.66)

def nivel_cltv(cltv):
    if cltv > p66: return 'ALTO'
    if cltv > p33: return 'MEDIO'
    return 'BAJO'

clientes_cltv['cltv_nivel'] = clientes_cltv['CLTV'].apply(nivel_cltv)

print(f"Percentiles CLTV: P33={p33:,.0f}€  |  P66={p66:,.0f}€")
print(f"\nDistribución de niveles CLTV:")
for nivel in ['BAJO', 'MEDIO', 'ALTO']:
    n = (clientes_cltv['cltv_nivel'] == nivel).sum()
    cltv_med = clientes_cltv[clientes_cltv['cltv_nivel'] == nivel]['CLTV'].median()
    print(f"  {nivel:5s}: {n:5,} clientes  |  CLTV mediano: {cltv_med:,.0f}€")

print(f"\nEjemplos representativos por segmento de churn:")
for seg in ['MUY_ALTO', 'ALTO', 'MEDIO', 'BAJO', 'MUY_BAJO']:
    ej = clientes_cltv[clientes_cltv['segmento'] == seg][['Modelo','prob_churn','C_n','CLTV','cltv_nivel']].head(2)
    print(f"\n  {seg}:")
    print(ej.to_string(index=False))

Percentiles CLTV: P33=113€  |  P66=197€

Distribución de niveles CLTV:
  BAJO : 3,300 clientes  |  CLTV mediano: 90€
  MEDIO: 3,314 clientes  |  CLTV mediano: 149€
  ALTO : 3,386 clientes  |  CLTV mediano: 330€

Ejemplos representativos por segmento de churn:

  MUY_ALTO:
Modelo  prob_churn    C_n  CLTV cltv_nivel
     I        0.82 408.10 57.03       BAJO
     A        0.82 267.50 36.74       BAJO

  ALTO:
Modelo  prob_churn    C_n   CLTV cltv_nivel
     A        0.64 267.50  95.24       BAJO
     H        0.65 388.30 135.59      MEDIO

  MEDIO:
Modelo  prob_churn    C_n   CLTV cltv_nivel
     B        0.59 281.41 129.65      MEDIO
     B        0.57 281.41 138.90      MEDIO

  BAJO:
Modelo  prob_churn    C_n     CLTV cltv_nivel
     H        0.22 388.30 1,218.88       ALTO
     I        0.27 408.10   939.77       ALTO

  MUY_BAJO:
Modelo  prob_churn    C_n     CLTV cltv_nivel
     D        0.05 319.00 5,949.92       ALTO
     I        0.09 408.10 4,601.26       ALTO


In [279]:
# ══════════════════════════════════════════════════════════════════════════════
# Asignación de estrategia 2D: Churn × CLTV
# ══════════════════════════════════════════════════════════════════════════════
# CLTV ALTO  → subir ligeramente la oferta
# CLTV MEDIO → oferta estándar, pequeñas reducciones
# CLTV BAJO  → reducir inversión (email básico, bonos mínimos)
#
# Segmentos BAJO y MUY_BAJO: se ofrece el Pack 5 revisiones con descuento
#   + bono 1.000€ para compra de futuro vehiculo (fidelizacion a largo plazo)
# ──────────────────────────────────────────────────────────────────────────────

PACK_5_REV = " + Pack 5 rev. (-10%) + 1.000€ prox. vehiculo"

ESTRATEGIA_2D = {
    # (churn_seg, cltv_nivel): (contacto€, adicional€, bono€, RR, descripcion)

    # MUY_ALTO: churn >80%, inversion minima
    ('MUY_ALTO', 'BAJO'):  (0.20,  0.00, 10.00, 0.08, 'Email + Bono 10€'),
    ('MUY_ALTO', 'MEDIO'): (0.20,  0.00, 15.00, 0.09, 'Email + Bono 15€'),
    ('MUY_ALTO', 'ALTO'):  (0.20, 15.00, 25.00, 0.12, 'Email + Lavado + Bono 25€'),

    # ALTO: churn 60-80%, oferta escalada por valor
    ('ALTO', 'BAJO'):  (0.20,  0.00, 10.00, 0.40, 'Email + Bono 10€'),
    ('ALTO', 'MEDIO'): (0.10, 15.00, 45.00, 0.80, 'Email + Lavado + Bono 45€'),
    ('ALTO', 'ALTO'):  (0.10, 65.00, 55.00, 0.85, 'Email + Pack (Lav+Neum) + Bono 55€'),

    # MEDIO: churn 40-60%, palanca emocional
    ('MEDIO', 'BAJO'):  (0.20,  0.00,  8.00, 0.25, 'Email + Bono 8€'),
    ('MEDIO', 'MEDIO'): (0.50, 30.00, 25.00, 0.55, 'SMS + Lavado + Bono 25€'),
    ('MEDIO', 'ALTO'):  (0.50, 36.00, 35.00, 0.60, 'SMS + Regalo + Lavado + Bono 35€'),

    # BAJO: churn 20-40%, fidelizacion a largo plazo
    ('BAJO', 'BAJO'):  (0.20,  0.00,  0.00, 0.10, 'Email recordatorio revision' + PACK_5_REV),
    ('BAJO', 'MEDIO'): (0.50,  0.00, 10.00, 0.45, 'SMS + Bono 10€' + PACK_5_REV),
    ('BAJO', 'ALTO'):  (0.50,  0.00, 25.00, 0.50, 'SMS + Bono 25€' + PACK_5_REV),

    # MUY_BAJO: churn <20%, retencion garantizada
    ('MUY_BAJO', 'BAJO'):  (0.20,  0.00,  0.00, 0.05, 'Email recordatorio revision' + PACK_5_REV),
    ('MUY_BAJO', 'MEDIO'): (0.20,  0.00, 10.00, 0.40, 'Email + Bono 10€' + PACK_5_REV),
    ('MUY_BAJO', 'ALTO'):  (0.20,  0.00, 15.00, 0.45, 'Email + Bono 15€' + PACK_5_REV),
}

def asignar_estrategia_cltv(row):
    return ESTRATEGIA_2D.get((row['segmento'], row['cltv_nivel']), (0.20, 0.00, 10.00, 0.20, 'Estandar'))

estrategias = clientes_cltv.apply(asignar_estrategia_cltv, axis=1, result_type='expand')
estrategias.columns = ['CC_cltv', 'CA_cltv', 'Bono_cltv', 'RR_cltv', 'Accion_cltv']
clientes_cltv = pd.concat([
    clientes_cltv.drop(columns=['CC_cltv','CA_cltv','Bono_cltv','RR_cltv','Accion_cltv'], errors='ignore'),
    estrategias
], axis=1)

print("=" * 130)
print("MATRIZ ESTRATEGICA 2D: Churn x CLTV")
print("=" * 130)
filas = []
for seg in ['MUY_ALTO', 'ALTO', 'MEDIO', 'BAJO', 'MUY_BAJO']:
    for nivel in ['BAJO', 'MEDIO', 'ALTO']:
        cc, ca, bono, rr, accion = ESTRATEGIA_2D[(seg, nivel)]
        n = ((clientes_cltv['segmento'] == seg) & (clientes_cltv['cltv_nivel'] == nivel)).sum()
        filas.append({'Churn': seg, 'CLTV': nivel, 'N': n,
                      'Contacto€': cc, 'Adicional€': ca, 'Bono€': bono,
                      'RR%': int(rr*100), 'Accion': accion})

df_matriz = pd.DataFrame(filas)
print(df_matriz[['Churn','CLTV','N','Contacto€','Adicional€','Bono€','RR%']].to_string(index=False))
print()
print("Acciones:")
for _, r in df_matriz.iterrows():
    print(f"  {r['Churn']:8s} x {r['CLTV']:5s} -> {r['Accion']}")
print("=" * 130)

print("\nPACK 5 REVISIONES (oferta exclusiva segmentos BAJO y MUY_BAJO):")
print("  - Precio: 5 revisiones con ~10% de descuento sobre tarifa individual")
print("  - Incluye: bono de 1.000€ aplicable a la compra del proximo vehiculo")
print("  - Objetivo: asegurar fidelidad durante los proximos 3-5 anos")
print("  - Response rate estimado del pack: ~20-30% de los que reciben la oferta")


MATRIZ ESTRATEGICA 2D: Churn x CLTV
   Churn  CLTV    N  Contacto€  Adicional€  Bono€  RR%
MUY_ALTO  BAJO  260       0.20        0.00  10.00    8
MUY_ALTO MEDIO    0       0.20        0.00  15.00    9
MUY_ALTO  ALTO    0       0.20       15.00  25.00   12
    ALTO  BAJO 3040       0.20        0.00  10.00   40
    ALTO MEDIO 1303       0.10       15.00  45.00   80
    ALTO  ALTO    0       0.10       65.00  55.00   85
   MEDIO  BAJO    0       0.20        0.00   8.00   25
   MEDIO MEDIO 2011       0.50       30.00  25.00   55
   MEDIO  ALTO 1905       0.50       36.00  35.00   60
    BAJO  BAJO    0       0.20        0.00   0.00   10
    BAJO MEDIO    0       0.50        0.00  10.00   45
    BAJO  ALTO   31       0.50        0.00  25.00   50
MUY_BAJO  BAJO    0       0.20        0.00   0.00    5
MUY_BAJO MEDIO    0       0.20        0.00  10.00   40
MUY_BAJO  ALTO 1450       0.20        0.00  15.00   45

Acciones:
  MUY_ALTO x BAJO  -> Email + Bono 10€
  MUY_ALTO x MEDIO -> Email + Bono

In [280]:
# ══════════════════════════════════════════════════════════════════════════════
# ROI con estrategia CLTV (por celda de la matriz 2D)
# ══════════════════════════════════════════════════════════════════════════════

resultados_cltv = []

for seg in ['MUY_ALTO', 'ALTO', 'MEDIO', 'BAJO', 'MUY_BAJO']:
    for nivel in ['BAJO', 'MEDIO', 'ALTO']:
        grupo = clientes_cltv[
            (clientes_cltv['segmento'] == seg) &
            (clientes_cltv['cltv_nivel'] == nivel)
        ]
        if len(grupo) == 0:
            continue

        cc, ca, bono, rr, accion = ESTRATEGIA_2D[(seg, nivel)]
        n_total = len(grupo)
        n_responden = n_total * rr

        coste_contacto_total  = n_total * cc
        coste_adicional_total = n_responden * ca
        coste_marketing_total = n_responden * grupo['Coste_Marketing'].mean()
        coste_bonos           = n_responden * bono

        costes_campana   = coste_contacto_total + coste_adicional_total + coste_marketing_total + coste_bonos
        margen_bruto_total = n_responden * grupo['Margen_Bruto'].mean()
        beneficio_neto   = margen_bruto_total - costes_campana
        roi = (beneficio_neto / costes_campana) if costes_campana > 0 else 0

        resultados_cltv.append({
            'Churn': seg, 'CLTV': nivel, 'N': n_total,
            'RR%': int(rr * 100), 'N_Resp': int(n_responden),
            'Costes€': costes_campana,
            'Margen_Bruto€': margen_bruto_total,
            'Beneficio€': beneficio_neto,
            'ROI': roi
        })

df_roi_cltv = pd.DataFrame(resultados_cltv)

costes_total_cltv   = df_roi_cltv['Costes€'].sum()
beneficio_total_cltv = df_roi_cltv['Beneficio€'].sum()
roi_global_cltv = beneficio_total_cltv / costes_total_cltv if costes_total_cltv > 0 else 0

print("=" * 120)
print("ROI POR CELDA DE LA MATRIZ 2D (Churn × CLTV)")
print("=" * 120)
print(df_roi_cltv.to_string(index=False))
print("=" * 120)
print(f"{'ROI GLOBAL (CLTV):':40s} {roi_global_cltv:.4f}")
print(f"{'Inversión Total Campaña:':40s} {costes_total_cltv:,.2f}€")
print(f"{'Beneficio Neto Total:':40s} {beneficio_total_cltv:,.2f}€")
print(f"{'Clientes contactados:':40s} {df_roi_cltv['N'].sum():,}")
print(f"{'Clientes que responden (est.):':40s} {int(df_roi_cltv['N_Resp'].sum()):,}")
print("=" * 120)

ROI POR CELDA DE LA MATRIZ 2D (Churn × CLTV)
   Churn  CLTV    N  RR%  N_Resp   Costes€  Margen_Bruto€  Beneficio€   ROI
MUY_ALTO  BAJO  260    8      20    334.73       4,633.12    4,298.40 12.84
    ALTO  BAJO 3040   40    1216 16,420.83     226,475.63  210,054.79 12.79
    ALTO MEDIO 1303   80    1042 66,284.13     223,809.31  157,525.18  2.38
   MEDIO MEDIO 2011   55    1106 65,249.67     211,507.78  146,258.12  2.24
   MEDIO  ALTO 1905   60    1143 86,611.01     279,341.70  192,730.69  2.23
    BAJO  ALTO   31   50      15    464.37       3,805.22    3,340.84  7.19
MUY_BAJO  ALTO 1450   45     652 12,275.18     136,256.13  123,980.95 10.10
ROI GLOBAL (CLTV):                       3.3847
Inversión Total Campaña:                 247,639.92€
Beneficio Neto Total:                    838,188.97€
Clientes contactados:                    10,000
Clientes que responden (est.):           5,194


In [281]:
# Proyeccion de revisiones futuras por modelo
# C(n) = BASE x (1+alfa)^n  |  Beneficio = C(n) x 0.62

N_MAX = 7
DESCUENTO_PACK = 0.10
BONO_VEHICULO  = 1000.0

modelos_info = (clientes_cltv
    .groupby('Modelo')['Mantenimiento_medio']
    .first().sort_index().to_dict()
)

# Churn medio real por modelo
churn_por_modelo = clientes_cltv.groupby('Modelo')['prob_churn'].mean().round(3)

# Tabla pivot: filas=Modelo, columnas=Rev 1..7 (beneficio no acumulado)
pivot_rows = []
pack_rows  = []

for modelo, base in modelos_info.items():
    alpha  = 0.07 if modelo in ['A', 'B'] else 0.10
    n_cli  = (clientes_cltv['Modelo'] == modelo).sum()
    p_churn = churn_por_modelo[modelo]

    row = {'Modelo': modelo, 'N clientes': n_cli, 'Churn medio': p_churn}
    for n in range(1, N_MAX + 1):
        benef = base * ((1 + alpha) ** n) * 0.62
        row['Rev ' + str(n)] = round(benef, 1)
    pivot_rows.append(row)

    ingreso_5   = sum(base * ((1 + alpha) ** n) for n in range(1, 6))
    precio_pack = ingreso_5 * (1 - DESCUENTO_PACK)
    benef_pack  = precio_pack * 0.62
    pack_rows.append({
        'Modelo': modelo,
        'Ingreso 5 rev. €': round(ingreso_5, 2),
        'Precio pack (-10%) €': round(precio_pack, 2),
        'Benef. taller pack €': round(benef_pack, 2),
        'Tras bono vehiculo €': round(benef_pack - BONO_VEHICULO, 2),
    })

df_pivot = pd.DataFrame(pivot_rows)
df_pack  = pd.DataFrame(pack_rows)

# Ordenar por churn medio (los que mas se quedan primero)
# ordenado A-K (orden natural del dict)

print("=" * 100)
print("BENEFICIO POR REVISION Y MODELO (no acumulado, en euros)")
print("Ordenado de Modelo A a Modelo K")
print("=" * 100)
print(df_pivot.to_string(index=False))

print()
print("=" * 80)
print("PACK 5 REVISIONES POR MODELO")
print("=" * 80)
print(df_pack.to_string(index=False))


BENEFICIO POR REVISION Y MODELO (no acumulado, en euros)
Ordenado de Modelo A a Modelo K
Modelo  N clientes  Churn medio  Rev 1  Rev 2  Rev 3  Rev 4  Rev 5  Rev 6  Rev 7
     A        1277         0.61 165.80 177.50 189.90 203.20 217.40 232.60 248.90
     B        2779         0.54 174.50 186.70 199.80 213.70 228.70 244.70 261.80
     C         705         0.50 188.20 207.10 227.80 250.50 275.60 303.10 333.50
     D        1547         0.49 197.80 217.60 239.30 263.20 289.60 318.50 350.40
     E         118         0.55 208.00 228.80 251.70 276.90 304.50 335.00 368.50
     F           2         0.59 218.20 240.10 264.10 290.50 319.50 351.50 386.60
     G           7         0.63 229.20 252.10 277.30 305.00 335.50 369.10 406.00
     H         473         0.69 240.70 264.80 291.30 320.40 352.50 387.70 426.50
     I        2625         0.53 253.00 278.30 306.20 336.80 370.40 407.50 448.20
     J         451         0.33 266.00 292.60 321.80 354.00 389.40 428.40 471.20
     K          16  

In [282]:
# Curva de beneficio neto por revision y modelo

# Construir df largo: una fila por (modelo, revision)
filas_rev = []
for modelo, base in modelos_info.items():
    alpha = 0.07 if modelo in ['A', 'B'] else 0.10
    for n in range(1, N_MAX + 1):
        filas_rev.append({
            'Modelo':        modelo,
            'Revision':      n,
            'Beneficio neto': round(base * ((1 + alpha) ** n) * 0.62, 2),
        })

df_rev = pd.DataFrame(filas_rev)

# Paleta de colores para cada modelo
colores = [
    '#2ecc71','#3498db','#9b59b6','#e67e22','#e74c3c',
    '#1abc9c','#f39c12','#d35400','#27ae60','#2980b9','#8e44ad'
]

fig = go.Figure()

for i, modelo in enumerate(sorted(modelos_info.keys())):
    subset = df_rev[df_rev['Modelo'] == modelo]
    churn_m = round(churn_por_modelo[modelo] * 100, 1)
    fig.add_trace(go.Scatter(
        x=subset['Revision'],
        y=subset['Beneficio neto'],
        mode='lines+markers',
        name='Modelo ' + modelo + ' (churn ' + str(churn_m) + '%)',
        line=dict(color=colores[i % len(colores)], width=2),
        marker=dict(size=7),
    ))

# Linea vertical en revision 5: umbral del pack
fig.add_vline(
    x=5, line_dash='dash', line_color='#e74c3c', line_width=2,
    annotation_text='Rev. 5: umbral pack + dto. 2º vehiculo',
    annotation_position='top left',
    annotation_font_color='#e74c3c',
)

fig.update_layout(
    title=dict(
        text='<b>Evolucion del beneficio neto por revision y modelo</b>',
        font=dict(size=18, color='#2c3e50')
    ),
    xaxis=dict(title='Numero de revision', tickmode='linear', dtick=1),
    yaxis=dict(title='Beneficio neto (€)'),
    template='plotly_white',
    height=500,
    legend=dict(orientation='v', x=1.01, y=1),
)

fig.show()

In [283]:
# ══════════════════════════════════════════════════════════════════════════════
# Comparativa: Estrategia Solo-Churn vs Estrategia Churn × CLTV
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("COMPARATIVA GLOBAL: Solo-Churn vs Churn × CLTV")
print("=" * 80)
print(f"{'Métrica':<35} {'Solo-Churn':>15} {'Churn × CLTV':>15} {'Diferencia':>12}")
print("-" * 80)
print(f"{'Inversión total (€)':<35} {costes_total:>15,.0f} {costes_total_cltv:>15,.0f} {costes_total_cltv - costes_total:>+12,.0f}")
print(f"{'Beneficio neto (€)':<35} {beneficio_total:>15,.0f} {beneficio_total_cltv:>15,.0f} {beneficio_total_cltv - beneficio_total:>+12,.0f}")
print(f"{'ROI global':<35} {roi_global:>15.2f} {roi_global_cltv:>15.2f} {roi_global_cltv - roi_global:>+12.2f}")
print(f"{'Clientes que responden':<35} {int(df_roi['N_Responden'].sum()):>15,} {int(df_roi_cltv['N_Resp'].sum()):>15,} {int(df_roi_cltv['N_Resp'].sum()) - int(df_roi['N_Responden'].sum()):>+12,}")
print("=" * 80)

# ── Gráfico comparativo: ROI por segmento ─────────────────────────────────────
roi_churn_seg = df_roi.set_index('Segmento')['ROI']
roi_cltv_seg  = df_roi_cltv.groupby('Churn')['Beneficio€'].sum() / df_roi_cltv.groupby('Churn')['Costes€'].sum()

segmentos_orden = ['MUY_ALTO', 'ALTO', 'MEDIO', 'BAJO', 'MUY_BAJO']
roi_churn_vals = [roi_churn_seg.get(s, 0) for s in segmentos_orden]
roi_cltv_vals  = [roi_cltv_seg.get(s, 0) for s in segmentos_orden]

fig_comp = go.Figure()

fig_comp.add_trace(go.Bar(
    name='Solo-Churn (base)',
    x=segmentos_orden,
    y=roi_churn_vals,
    marker_color='#3498db',
    text=[f"{v:.2f}x" for v in roi_churn_vals],
    textposition='outside'
))

fig_comp.add_trace(go.Bar(
    name='Churn × CLTV (nuevo)',
    x=segmentos_orden,
    y=roi_cltv_vals,
    marker_color='#27ae60',
    text=[f"{v:.2f}x" for v in roi_cltv_vals],
    textposition='outside'
))

fig_comp.add_hline(y=1, line_dash='dash', line_color='red', line_width=1,
                   annotation_text='Punto de equilibrio', annotation_position='right')

fig_comp.update_layout(
    title=dict(
        text='<b>Comparativa ROI por Segmento: Solo-Churn vs Churn × CLTV</b>',
        font=dict(size=18, color='#2c3e50')
    ),
    xaxis_title='Segmento de Churn',
    yaxis_title='ROI',
    barmode='group',
    template='plotly_white',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig_comp.show()

# ── Heatmap de ROI de la matriz 2D ───────────────────────────────────────────
pivot_roi = df_roi_cltv.pivot(index='Churn', columns='CLTV', values='ROI').reindex(
    index=segmentos_orden, columns=['BAJO', 'MEDIO', 'ALTO']
).fillna(0)

fig_heat = go.Figure(go.Heatmap(
    z=pivot_roi.values,
    x=['CLTV Bajo', 'CLTV Medio', 'CLTV Alto'],
    y=segmentos_orden,
    colorscale='RdYlGn',
    text=[[f"{v:.1f}x" for v in row] for row in pivot_roi.values],
    texttemplate="%{text}",
    textfont=dict(size=14, color='black'),
    colorbar=dict(title='ROI')
))

fig_heat.update_layout(
    title=dict(
        text='<b>Heatmap de ROI: Matriz Churn × CLTV</b>',
        font=dict(size=18, color='#2c3e50')
    ),
    xaxis_title='Nivel CLTV',
    yaxis_title='Segmento Churn',
    template='plotly_white',
    height=450
)

fig_heat.show()

print("\n💡 CONCLUSIÓN:")
print("   La estrategia Churn × CLTV permite asignar la inversión de forma más eficiente:")
print("   - Clientes de alto valor (CLTV ALTO) reciben ofertas más agresivas independientemente del churn")
print("   - Clientes de bajo valor (CLTV BAJO) con churn bajo son descartados (sin acción)")
print("   - El resultado es un ROI ajustado que optimiza el retorno por euro invertido")


COMPARATIVA GLOBAL: Solo-Churn vs Churn × CLTV
Métrica                                  Solo-Churn    Churn × CLTV   Diferencia
--------------------------------------------------------------------------------
Inversión total (€)                         693,219         247,640     -445,579
Beneficio neto (€)                          581,446         838,189     +256,743
ROI global                                     0.84            3.38        +2.55
Clientes que responden                        6,246           5,194       -1,052



💡 CONCLUSIÓN:
   La estrategia Churn × CLTV permite asignar la inversión de forma más eficiente:
   - Clientes de alto valor (CLTV ALTO) reciben ofertas más agresivas independientemente del churn
   - Clientes de bajo valor (CLTV BAJO) con churn bajo son descartados (sin acción)
   - El resultado es un ROI ajustado que optimiza el retorno por euro invertido
